# LLM 14: Multi-Modal Basics

Hands-on:
1. Send an image to Claude/GPT-4o and get structured extraction
2. Transcribe audio with Whisper
3. Measure image token cost
4. CLIP-style text↔image similarity (local)
5. Exercises: multi-modal RAG sketch, chart-to-code

Deps: `pip install anthropic openai pillow torch transformers`

API keys optional; cells skip gracefully when missing.

## 1. Create a Test Image

Generate a simple invoice-style image we can send to a vision model.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import base64, io

def make_invoice():
    img = Image.new('RGB', (600, 400), 'white')
    d = ImageDraw.Draw(img)
    lines = [
        'NovaCore Industries',
        'Invoice #2026-04-0042',
        '',
        'Widget Pro x 3   @ $25.00   = $75.00',
        'Adapter Cable x 5 @ $4.50   = $22.50',
        'Shipping                    = $12.00',
        '-----------------------------',
        'Total                       = $109.50',
    ]
    y = 30
    for ln in lines:
        d.text((30, y), ln, fill='black')
        y += 30
    return img

img = make_invoice()
img.save('invoice.png')
display(img)

## 2. Structured Extraction via Vision

In [ ]:
import os, base64, json, io

def img_to_b64(pil_img, fmt='PNG'):
    buf = io.BytesIO(); pil_img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()

if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic()
    b64 = img_to_b64(img)
    r = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=400,
        messages=[{
            'role':'user',
            'content':[
                {'type':'image','source':{'type':'base64','media_type':'image/png','data':b64}},
                {'type':'text','text':'Extract vendor, invoice_number, line_items (desc/qty/unit_price/subtotal), shipping, total. Return JSON only.'},
            ],
        }],
    )
    print(r.content[0].text)
elif os.getenv('OPENAI_API_KEY'):
    from openai import OpenAI
    client = OpenAI()
    b64 = img_to_b64(img)
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role':'user','content':[
            {'type':'text','text':'Extract vendor, invoice_number, line_items (desc/qty/unit_price/subtotal), shipping, total. Return JSON.'},
            {'type':'image_url','image_url':{'url':f'data:image/png;base64,{b64}'}},
        ]}],
    )
    print(r.choices[0].message.content)
else:
    print('Skipped (no vision-capable API key).')

## 3. CLIP-Style Text↔Image Similarity (Local)

Useful for multi-modal retrieval without API calls.

In [ ]:
try:
    from transformers import CLIPProcessor, CLIPModel
    import torch
    model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

    prompts = ['an invoice', 'a photo of a dog', 'a chart', 'handwritten notes']
    inputs = proc(text=prompts, images=img, return_tensors='pt', padding=True)
    with torch.no_grad():
        out = model(**inputs)
    logits = out.logits_per_image[0].softmax(dim=-1).tolist()
    for p, s in sorted(zip(prompts, logits), key=lambda x: -x[1]):
        print(f'{s:.3f}  {p}')
except Exception as e:
    print('Skipped CLIP demo:', e)

## 4. Whisper for Audio Transcription (Local)

In [ ]:
# Generate a tiny audio sample we can transcribe (synthesized sine-wave speech placeholder).
# Replace with a real .wav file for meaningful output.
#
# try:
#     import whisper
#     w = whisper.load_model('tiny')
#     result = w.transcribe('sample.wav')
#     print(result['text'])
# except Exception as e:
#     print('Skipped:', e)

print('See comment block for a runnable snippet with `pip install openai-whisper` and a real .wav file.')

## 5. Image Token Cost Estimator

In [ ]:
def image_tokens_estimate(width, height, provider='claude'):
    # Rough, illustrative formulas — always confirm in provider docs.
    if provider == 'claude':
        return int((width * height) / 750)
    if provider == 'openai-high':
        tiles = ((width + 511) // 512) * ((height + 511) // 512)
        return 85 + 170 * tiles
    if provider == 'openai-low':
        return 85
    return int((width * height) / 600)

for w, h in [(512, 512), (1024, 1024), (2048, 1536), (1920, 1080)]:
    print(f'{w}x{h}:',
          f'claude~{image_tokens_estimate(w, h, "claude"):>5} tok,',
          f'openai-high~{image_tokens_estimate(w, h, "openai-high"):>5} tok,',
          f'openai-low~{image_tokens_estimate(w, h, "openai-low"):>4} tok')

## 6. Exercise: Multi-Modal RAG Sketch

Build a mini retrieval pipeline:
1. Embed 10 images with CLIP; store vectors.
2. Embed 10 text snippets with a sentence-transformer.
3. Given a text query, retrieve (a) top-3 images via CLIP text encoder, (b) top-3 texts via sentence-transformer.
4. Send retrieved images + texts to a vision-language model to answer the query.

This is the shape of real multi-modal RAG — dual indexes, modality-appropriate retrievers, a VL synthesizer at the end.

## 7. Exercise: Chart-to-Code

Given a screenshot of a bar chart (bring your own), ask a VL model to return matplotlib code that reproduces it. Execute the code; compare to the original.

Real-world usage: converting legacy PDFs into re-runnable analyses, reverse-engineering dashboards, generating docs.

## 8. Takeaways

- **Every strong 2026 model is multi-modal.** Plan for it.
- **Images are expensive tokens.** Extract to text if possible.
- **Realtime voice is a separate architecture** from ASR→LLM→TTS.
- **Structured output + vision** is the killer app for document AI.
- **CLIP-class models** power practical multi-modal retrieval locally.

Next: **15 — Hands-On: Claude + OpenAI + Local**, where we build one provider abstraction to run the same prompt across three stacks.